In [24]:
# ==========================================================
# HIGH RISK PREGNANCY EXPLAINABLE AI SYSTEM
# ==========================================================
#
# Project:
# Explainable AI Framework for High-Risk Pregnancy Prediction
#
# Purpose:
# --------
# This notebook demonstrates an Explainable AI (XAI)
# pipeline built on top of a pretrained Random Forest model.
#
# Workflow:
#
# Patient Data
#       ↓
# Random Forest Prediction
#       ↓
# SHAP Explanation
#       ↓
# Risk Mapping
#       ↓
# Possible Pregnancy Risk Categories
#       ↓
# Patient-Friendly Report
#
# ==========================================================


# ==========================================================
# Import Required Libraries
# ==========================================================

import joblib                 # Load trained Random Forest model
import shap                   # Explainable AI
import pandas as pd           # Data manipulation
import numpy as np            # Numerical computations

In [25]:
# ==========================================================
# Load Trained Random Forest Model
# ==========================================================
#
# Purpose:
# --------
# Loads the pretrained Random Forest model from disk.
#
# Input:
# ------
# model_path : str
#
# Output:
# -------
# model : Trained Random Forest Classifier
#
# ==========================================================

def load_prediction_model(model_path):

    model = joblib.load(model_path)

    return model

In [26]:
# ==========================================================
# Load Model
# ==========================================================

MODEL_PATH = "random_forest_model.pkl"

model = load_prediction_model(MODEL_PATH)

print("Model loaded successfully!")
print(model)

Model loaded successfully!
RandomForestClassifier()


C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\hp\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [27]:
# ==========================================================
# Prediction Engine
# ==========================================================
#
# Purpose
# -------
# Predict whether a pregnancy is High Risk or Low Risk
# using the pretrained Random Forest model.
#
# Inputs
# ------
# model   : Trained Random Forest model
# patient : DataFrame containing one patient's clinical data
#
# Outputs
# -------
# prediction  : 0 (Low Risk) or 1 (High Risk)
# probability : Prediction probabilities for both classes
#
# ==========================================================

def predict_risk(model, patient):

    # Predict class
    prediction = model.predict(patient)

    # Predict probability
    probability = model.predict_proba(patient)

    return prediction, probability

In [28]:
# ==========================================================
# Sample Patient (Testing Only)
# ==========================================================
#
# NOTE:
# This patient is only used to test the XAI pipeline.
#
# In the website, these values will come directly
# from the user interface.
#
# ==========================================================

patient = pd.DataFrame([{

    "Age": 39,
    "G": 5,
    "P": 4,
    "L": 4,
    "A": 2,
    "D": 0,

    "SystolicBP": 180,
    "DiastolicBP": 120,

    "RBS": 220,

    "BodyTemp": 101,

    "HeartRate": 118,

    "HB": 7.5,

    "HBA1C": 8.5,

    "RR": 28

}])

In [29]:
# ==========================================================
# Predict Pregnancy Risk
# ==========================================================

prediction, probability = predict_risk(model, patient)

print("=" * 50)
print("PREGNANCY RISK PREDICTION")
print("=" * 50)

if prediction[0] == 1:
    print("Prediction : HIGH RISK")
else:
    print("Prediction : LOW RISK")

print(f"\nPrediction Probabilities:\n{probability}")

PREGNANCY RISK PREDICTION
Prediction : HIGH RISK

Prediction Probabilities:
[[0. 1.]]


In [30]:
# ==========================================================
# SHAP Engine
# ==========================================================
#
# Purpose
# -------
# Create a SHAP TreeExplainer for the trained
# Random Forest model.
#
# The explainer understands how the model makes
# predictions and will later explain individual
# patient predictions.
#
# Input
# -----
# model : Trained Random Forest model
#
# Output
# ------
# explainer : SHAP TreeExplainer object
#
# ==========================================================

def create_shap_explainer(model):

    explainer = shap.TreeExplainer(model)

    return explainer

In [31]:
# ==========================================================
# Generate SHAP Values
# ==========================================================
#
# Purpose
# -------
# Generate SHAP values for an individual patient.
#
# SHAP values quantify how much each feature
# contributed to the prediction.
#
# Inputs
# ------
# explainer : SHAP explainer object
# patient   : Patient DataFrame
#
# Output
# ------
# shap_values : SHAP explanation object
#
# ==========================================================

def generate_shap_values(explainer, patient):

    shap_values = explainer(patient)

    return shap_values

In [32]:
# ==========================================================
# Generate SHAP Explanation
# ==========================================================

explainer = create_shap_explainer(model)

shap_values = generate_shap_values(explainer, patient)

print("SHAP values generated successfully!")

SHAP values generated successfully!


In [33]:
# ==========================================================
# Feature Ranking Engine
# ==========================================================
#
# Purpose
# -------
# Create a ranked table of SHAP contributions
# for the patient's prediction.
#
# Inputs
# ------
# patient     : Patient DataFrame
# shap_values : SHAP explanation object
#
# Output
# ------
# shap_df : DataFrame containing
#           Feature Name
#           Patient Value
#           SHAP Value
#           Absolute SHAP Value
#
# ==========================================================

def create_shap_dataframe(patient, shap_values):

    # SHAP values for the High Risk class (Class 1)
    high_risk_shap = shap_values.values[0, :, 1]

    shap_df = pd.DataFrame({

        "Feature": patient.columns,

        "Patient Value": patient.iloc[0].values,

        "SHAP Value": high_risk_shap

    })

    # Calculate absolute contribution
    shap_df["Absolute SHAP"] = shap_df["SHAP Value"].abs()

    # Sort features by importance
    shap_df = shap_df.sort_values(
        by="Absolute SHAP",
        ascending=False
    ).reset_index(drop=True)

    return shap_df

In [34]:
# ==========================================================
# Rank Features by SHAP Importance
# ==========================================================

shap_df = create_shap_dataframe(patient, shap_values)

shap_df

,Feature,Patient Value,SHAP Value,Absolute SHAP
0,RBS,220.0,0.132746,0.132746
1,G,5.0,0.075481,0.075481
2,A,2.0,0.058904,0.058904
3,HB,7.5,0.056886,0.056886
4,L,4.0,0.026005,0.026005
5,BodyTemp,101.0,0.023198,0.023198
6,P,4.0,0.021065,0.021065
7,Age,39.0,0.019664,0.019664
8,SystolicBP,180.0,0.015390,0.015390
9,DiastolicBP,120.0,0.014501,0.014501


In [35]:
# ==========================================================
# Pregnancy Risk Knowledge Base
# ==========================================================
#
# PURPOSE
# -------
# The Random Forest model predicts whether a pregnancy
# is HIGH RISK or LOW RISK.
#
# SHAP identifies the clinical parameters that contributed
# most to the prediction.
#
# This knowledge base maps those SHAP-important parameters
# to possible pregnancy risk categories.
#
# IMPORTANT
# ---------
# This knowledge base DOES NOT diagnose pregnancy
# complications.
#
# It is only used to identify the pregnancy complications
# that are most consistent with the SHAP explanation.
#
# ==========================================================

RISK_KNOWLEDGE_BASE = {

    # ------------------------------------------------------
    # Hypertensive Disorders of Pregnancy
    # ------------------------------------------------------
    "Hypertensive Disorders of Pregnancy": {

        "features": [
            "SystolicBP",
            "DiastolicBP",
            "Age"
        ]

    },

    # ------------------------------------------------------
    # Gestational Diabetes Mellitus (GDM)
    # ------------------------------------------------------
    "Gestational Diabetes Mellitus (GDM)": {

        "features": [
            "RBS",
            "HBA1C",
            "Age"
        ]

    },

    # ------------------------------------------------------
    # Maternal Anemia
    # ------------------------------------------------------
    "Maternal Anemia": {

        "features": [
            "HB",
            "HeartRate",
            "RR"
        ]

    },

    # ------------------------------------------------------
    # Preterm Birth
    # ------------------------------------------------------
    "Preterm Birth": {

        "features": [
            "Age",
            "G",
            "P",
            "A",
            "HB",
            "SystolicBP",
            "DiastolicBP"
        ]

    },

    # ------------------------------------------------------
    # Fetal Growth Restriction (FGR)
    # ------------------------------------------------------
    "Fetal Growth Restriction (FGR)": {

        "features": [
            "HB",
            "SystolicBP",
            "DiastolicBP"
        ]

    },

    # ------------------------------------------------------
    # Maternal Infection / Sepsis
    # ------------------------------------------------------
    "Maternal Infection / Sepsis": {

        "features": [
            "BodyTemp",
            "HeartRate",
            "RR"
        ]

    }

}

In [45]:
# ==========================================================
# Risk Mapping Engine
# ==========================================================
#
# PURPOSE
# -------
# Map the SHAP-important clinical parameters to
# possible pregnancy risk categories.
#
# The contribution of each feature is weighted
# using its SHAP value.
#
# INPUT
# -----
# shap_df
# RISK_KNOWLEDGE_BASE
#
# OUTPUT
# ------
# Ranked pregnancy risk categories
#
# ==========================================================

def map_risks(shap_df, knowledge_base):

    risk_scores = {}

    # Iterate through each pregnancy risk
    for risk_name, risk_info in knowledge_base.items():

        score = 0
        matched_features = []

        # Features associated with this pregnancy risk
        risk_features = risk_info["features"]

        # Check every SHAP feature
        for _, row in shap_df.iterrows():

            feature = row["Feature"]
            shap_value = abs(row["SHAP Value"])

            if feature in risk_features:

                score += shap_value

                matched_features.append(feature)

        risk_scores[risk_name] = {

            "Risk Score": score,

            "Matched Features": matched_features

        }

    return risk_scores

In [46]:
# ==========================================================
# Map SHAP Features to Pregnancy Risk Categories
# ==========================================================

###  RISK_KNOWLEDGE_BASE
##risk_scores

In [47]:
# ==========================================================
# Create Risk Ranking Table
# ==========================================================
#
# Purpose
# -------
# Convert the risk mapping dictionary into a
# clean and readable table.
#
# This table can later be displayed directly
# on the website.
#
# ==========================================================

def create_risk_table(risk_scores):

    risk_table = []

    # Convert dictionary into table rows
    for risk, info in risk_scores.items():

        risk_table.append({

            "Risk Category": risk,

            "Risk Score": round(info["Risk Score"], 4),

            "Matched Features": ", ".join(info["Matched Features"]),

            "No. of Features Matched": len(info["Matched Features"])

        })

    # Create DataFrame
    risk_table = pd.DataFrame(risk_table)

    # Sort by highest score
    risk_table = risk_table.sort_values(
        by="Risk Score",
        ascending=False
    ).reset_index(drop=True)

    return risk_table

In [48]:
# ==========================================================
# Display Ranked Pregnancy Risks
# ==========================================================

risk_table = create_risk_table(risk_scores)

risk_table

,Risk Category,Risk Score,Matched Features,No. of Features Matched
0,Preterm Birth,0.2619,"G, A, HB, P, Age, SystolicBP, DiastolicBP",7
1,Gestational Diabetes Mellitus (GDM),0.1585,"RBS, Age, HBA1C",3
2,Fetal Growth Restriction (FGR),0.0868,"HB, SystolicBP, DiastolicBP",3
3,Maternal Anemia,0.0640,"HB, RR, HeartRate",3
4,Hypertensive Disorders of Pregnancy,0.0496,"Age, SystolicBP, DiastolicBP",3
5,Maternal Infection / Sepsis,0.0303,"BodyTemp, RR, HeartRate",3


In [49]:
# ==========================================================
# Select Top Pregnancy Risk Categories
# ==========================================================
#
# Purpose
# -------
# Select the pregnancy risk categories with the
# highest XAI Evidence Scores.
#
# These risks will later be displayed to the
# clinician and patient.
#
# Input
# -----
# risk_table
#
# Output
# ------
# top_risks
#
# ==========================================================

def get_top_risk_categories(risk_table, top_n=3):

    top_risks = risk_table.head(top_n).copy()

    return top_risks

In [50]:
# ==========================================================
# Display Top Pregnancy Risk Categories
# ==========================================================

top_risks = get_top_risk_categories(risk_table)

top_risks

,Risk Category,Risk Score,Matched Features,No. of Features Matched
0,Preterm Birth,0.2619,"G, A, HB, P, Age, SystolicBP, DiastolicBP",7
1,Gestational Diabetes Mellitus (GDM),0.1585,"RBS, Age, HBA1C",3
2,Fetal Growth Restriction (FGR),0.0868,"HB, SystolicBP, DiastolicBP",3


In [51]:
# ==========================================================
# AI Clinical Summary Generator (Patient-Friendly)
# ==========================================================

def generate_clinical_summary(prediction, risk_table):

    # Overall Prediction
    if prediction[0] == 1:
        prediction_text = "🔴 HIGH RISK PREGNANCY"
    else:
        prediction_text = "🟢 LOW RISK PREGNANCY"

    top_risks = risk_table.head(3)

    feature_map = {
        "HB": "Hemoglobin (blood level)",
        "RBS": "Blood sugar level",
        "HBA1C": "Long-term blood sugar",
        "Age": "Maternal age",
        "SystolicBP": "Upper blood pressure",
        "DiastolicBP": "Lower blood pressure",
        "HeartRate": "Heart rate",
        "RR": "Breathing rate",
        "BodyTemp": "Body temperature",
        "G": "Previous pregnancies",
        "P": "Previous births",
        "A": "Previous pregnancy losses",
        "L": "Number of living children",
        "D": "Previous child deaths"
    }

    summary = ""
    summary += "=" * 70 + "\n"
    summary += "              YOUR PREGNANCY HEALTH SUMMARY\n"
    summary += "=" * 70 + "\n\n"

    summary += f"Overall Assessment: {prediction_text}\n\n"

    if prediction[0] == 1:
        summary += (
            "💜 Please don't panic. This result does not mean that you "
            "or your baby will definitely have complications. It simply "
            "suggests that your pregnancy may benefit from closer medical "
            "monitoring and regular antenatal care.\n\n"
        )
    else:
        summary += (
            "💚 Based on the information provided, your pregnancy appears "
            "to be at lower risk. Continue your regular antenatal check-ups "
            "and follow your doctor's advice.\n\n"
        )

    summary += "The AI found the strongest evidence for:\n\n"

    medals = ["🥇", "🥈", "🥉"]

    for i, (_, row) in enumerate(top_risks.iterrows()):

        summary += f"{medals[i]} {row['Risk Category']}\n"

        features = [
            feature_map.get(f.strip(), f.strip())
            for f in row["Matched Features"].split(",")
        ]

        summary += "   Influenced by: "
        summary += ", ".join(features)
        summary += "\n\n"

    summary += (
        "🌸 This report is designed to support your healthcare provider. "
        "It is not a medical diagnosis, and many women with high-risk "
        "pregnancies go on to deliver healthy babies with timely care and "
        "regular follow-up."
    )

    return summary

In [52]:
# ==========================================================
# Generate AI Clinical Summary
# ==========================================================

summary = generate_clinical_summary(
    prediction,
    risk_table
)

print(summary)

              YOUR PREGNANCY HEALTH SUMMARY

Overall Assessment: 🔴 HIGH RISK PREGNANCY

💜 Please don't panic. This result does not mean that you or your baby will definitely have complications. It simply suggests that your pregnancy may benefit from closer medical monitoring and regular antenatal care.

The AI found the strongest evidence for:

🥇 Preterm Birth
   Influenced by: Previous pregnancies, Previous pregnancy losses, Hemoglobin (blood level), Previous births, Maternal age, Upper blood pressure, Lower blood pressure

🥈 Gestational Diabetes Mellitus (GDM)
   Influenced by: Blood sugar level, Maternal age, Long-term blood sugar

🥉 Fetal Growth Restriction (FGR)
   Influenced by: Hemoglobin (blood level), Upper blood pressure, Lower blood pressure

🌸 This report is designed to support your healthcare provider. It is not a medical diagnosis, and many women with high-risk pregnancies go on to deliver healthy babies with timely care and regular follow-up.
